# PesaPap Customer Growth, Retention and Value Analytics

## Notebook 3: SQL, KPI and Statistical Analysis

This notebook converts the cleaned exploratory findings into business evidence using:

- KPI definitions and scorecards
- SQL analysis with SQLite
- JOIN, GROUP BY, HAVING and window functions
- statistical testing
- business recommendations
- limitations and next steps


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

# Detect project root
cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    ROOT = cwd
elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    ROOT = cwd.parent
elif Path("/content/drive/MyDrive/plp-capstone-B-robin-kinyua").exists():
    ROOT = Path("/content/drive/MyDrive/plp-capstone-B-robin-kinyua")
else:
    raise FileNotFoundError(
        "Project root not found. Run from the repository root "
        "or notebooks folder."
    )

DATA = ROOT / "data"
PROCESSED = DATA / "processed"
SQL_DIR = ROOT / "sql"

SQL_DIR.mkdir(parents=True, exist_ok=True)

users = pd.read_csv(
    PROCESSED / "users_clean.csv",
    parse_dates=["signup_date"]
)

transactions = pd.read_csv(
    PROCESSED / "transactions_clean.csv",
    parse_dates=["txn_date"]
)

campaigns = pd.read_csv(
    PROCESSED / "campaigns_clean.csv"
)

activity = pd.read_csv(
    PROCESSED / "activity_clean.csv"
)

print("Users:", users.shape)
print("Transactions:", transactions.shape)
print("Campaigns:", campaigns.shape)
print("Activity:", activity.shape)


Users: (8000, 5)
Transactions: (70824, 7)
Campaigns: (12, 5)
Activity: (8000, 4)


## 1. KPI Definitions

The following KPIs are used to evaluate PesaPap customer growth, activity and transaction value.

**Total Customers:** Number of registered users.

**Active Customers:** Number of users who completed at least one valid transaction during the observation period.

**Active Rate:** Active customers divided by registered customers.

**Total Transactions:** Number of unique valid transactions.

**Total Transaction Value:** Sum of all valid transaction amounts.

**Average Transaction Value:** Total transaction value divided by total transactions.

**Average Customer Value:** Total transaction value divided by registered customers.


In [2]:
total_customers = users["user_id"].nunique()
active_customers = int(activity["active_flag"].sum())

active_rate = (
    active_customers / total_customers * 100
)

total_transactions = (
    transactions["txn_id"].nunique()
)

total_transaction_value = (
    transactions["amount"].sum()
)

average_transaction_value = (
    total_transaction_value / total_transactions
)

average_customer_value = (
    total_transaction_value / total_customers
)

kpi_scorecard = pd.DataFrame({
    "KPI": [
        "Total Customers",
        "Active Customers",
        "Active Rate (%)",
        "Total Transactions",
        "Total Transaction Value (KES)",
        "Average Transaction Value (KES)",
        "Average Customer Value (KES)"
    ],
    "Value": [
        total_customers,
        active_customers,
        round(active_rate, 2),
        total_transactions,
        round(total_transaction_value, 2),
        round(average_transaction_value, 2),
        round(average_customer_value, 2)
    ]
})

display(kpi_scorecard)


,KPI,Value
0,Total Customers,8.000000e+03
1,Active Customers,7.522000e+03
2,Active Rate (%),9.403000e+01
3,Total Transactions,7.082400e+04
4,Total Transaction Value (KES),1.111781e+08
5,Average Transaction Value (KES),1.569780e+03
6,Average Customer Value (KES),1.389726e+04


### KPI Interpretation

PesaPap has **8,000 registered customers**, of whom **7,522** completed at least one valid transaction during the observation period. This represents an ever-active customer rate of approximately **94.02%**.

The cleaned dataset contains **70,824 transactions** with a combined value of approximately **KES 111.18 million**.

Average transaction value is approximately **KES 1,569.78**, while average transaction value per registered customer is approximately **KES 13,897.26**.


## 2. SQLite Database

The clean tables generated in Notebook 1 are loaded into SQLite so the business questions can be answered using SQL.


In [3]:
db_path = DATA / "pesapap_capstone.db"

conn = sqlite3.connect(db_path)

users.to_sql(
    "users",
    conn,
    if_exists="replace",
    index=False
)

transactions.to_sql(
    "transactions",
    conn,
    if_exists="replace",
    index=False
)

campaigns.to_sql(
    "campaigns",
    conn,
    if_exists="replace",
    index=False
)

activity.to_sql(
    "activity",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite database ready:", db_path)


SQLite database ready: /content/drive/MyDrive/plp-capstone-B-robin-kinyua/data/pesapap_capstone.db


## 3. SQL Analysis

The SQL analysis includes more than five documented queries and demonstrates the required techniques:

- JOIN
- GROUP BY
- HAVING
- common table expressions
- window functions


### Query 1: Acquisition Channel Performance

This query joins customers to transactions and compares acquisition channels by customer count, transaction volume and transaction value.


In [4]:
query_1 = '''
SELECT
    u.acquisition_channel,
    COUNT(DISTINCT u.user_id) AS customers,
    COUNT(t.txn_id) AS transactions,
    ROUND(SUM(t.amount), 2) AS total_value,
    ROUND(AVG(t.amount), 2) AS average_transaction_value
FROM users AS u
LEFT JOIN transactions AS t
    ON u.user_id = t.user_id
GROUP BY u.acquisition_channel
ORDER BY total_value DESC;
'''

channel_sql = pd.read_sql_query(
    query_1,
    conn
)

display(channel_sql)


,acquisition_channel,customers,transactions,total_value,average_transaction_value
0,Referral,1942,18473,29028368.0,1571.39
1,Social,1818,15002,23555038.0,1570.13
2,Agent,1596,13645,21343379.0,1564.19
3,Search,1471,13312,20714640.0,1556.09
4,Partnership,1173,10392,16536680.0,1591.29


**Interpretation:** Referral is the strongest acquisition channel by total transaction value at approximately **KES 29.03 million**. Social is second at approximately **KES 23.56 million**.


### Query 2: Product Performance

This query compares products using transaction count, total transaction value and average transaction value.


In [5]:
query_2 = '''
SELECT
    product,
    COUNT(txn_id) AS transactions,
    ROUND(SUM(amount), 2) AS total_value,
    ROUND(AVG(amount), 2) AS average_transaction_value
FROM transactions
GROUP BY product
ORDER BY total_value DESC;
'''

product_sql = pd.read_sql_query(
    query_2,
    conn
)

display(product_sql)


,product,transactions,total_value,average_transaction_value
0,Send Money,21043,32978781.0,1567.21
1,Airtime,17192,26716946.0,1554.03
2,Bill Pay,12861,20296324.0,1578.13
3,Merchant Pay,12681,20169924.0,1590.56
4,Savings,7047,11016130.0,1563.24


**Interpretation:** Send Money is the leading product, generating approximately **KES 32.98 million** from **21,043 transactions**.


### Query 3: Monthly Transaction Trend with Window Functions

This query calculates monthly transaction value, cumulative value and month-over-month growth using `LAG()` and a running `SUM()`.


In [6]:
query_3 = '''
WITH monthly AS (
    SELECT
        strftime('%Y-%m', txn_date) AS month,
        COUNT(txn_id) AS transactions,
        SUM(amount) AS monthly_value
    FROM transactions
    GROUP BY strftime('%Y-%m', txn_date)
),
with_previous AS (
    SELECT
        month,
        transactions,
        monthly_value,
        LAG(monthly_value) OVER (
            ORDER BY month
        ) AS previous_value,
        SUM(monthly_value) OVER (
            ORDER BY month
        ) AS running_total
    FROM monthly
)
SELECT
    month,
    transactions,
    ROUND(monthly_value, 2) AS monthly_value,
    ROUND(running_total, 2) AS running_total,
    ROUND(
        (
            monthly_value - previous_value
        ) * 100.0 / previous_value,
        2
    ) AS mom_growth_pct
FROM with_previous
ORDER BY month;
'''

monthly_sql = pd.read_sql_query(
    query_3,
    conn
)

display(monthly_sql)


,month,transactions,monthly_value,running_total,mom_growth_pct
0,2025-07,1103,1729782.0,1729782.0,NaN
1,2025-08,2185,3416512.0,5146294.0,97.51
2,2025-09,3301,5147727.0,10294021.0,50.67
3,2025-10,3956,6471399.0,16765420.0,25.71
4,2025-11,5270,8342521.0,25107941.0,28.91
5,2025-12,5804,9179888.0,34287829.0,10.04
6,2026-01,6477,10319688.0,44607517.0,12.42
7,2026-02,7119,10854653.0,55462170.0,5.18
8,2026-03,8016,12644770.0,68106940.0,16.49
9,2026-04,8370,12717716.0,80824656.0,0.58


**Interpretation:** Monthly transaction value rises across the observation period. However, this should not be interpreted as pure organic growth because the synthetic dataset contains an expanding customer base and different observation windows for different signup cohorts.


### Query 4: County Performance Using a JOIN

This query compares counties by customer count, transaction value and average transaction value per registered customer.


In [7]:
query_4 = '''
SELECT
    u.county,
    COUNT(DISTINCT u.user_id) AS customers,
    COUNT(t.txn_id) AS transactions,
    ROUND(SUM(t.amount), 2) AS total_value,
    ROUND(
        SUM(t.amount) / COUNT(DISTINCT u.user_id),
        2
    ) AS value_per_customer
FROM users AS u
LEFT JOIN transactions AS t
    ON u.user_id = t.user_id
GROUP BY u.county
ORDER BY total_value DESC;
'''

county_sql = pd.read_sql_query(
    query_4,
    conn
)

display(county_sql)


,county,customers,transactions,total_value,value_per_customer
0,Nairobi,2425,21256,33307748.0,13735.15
1,Mombasa,1123,9723,15002566.0,13359.36
2,Kisumu,985,8719,13838053.0,14048.78
3,Nakuru,967,8790,13655325.0,14121.33
4,Machakos,909,8080,12789944.0,14070.35
5,Nyeri,806,7354,11826199.0,14672.70
6,Eldoret,785,6902,10758270.0,13704.80


**Interpretation:** Nairobi has the largest total transaction value because it has the largest customer population. Nyeri has the highest transaction value per registered customer, showing that market size and customer value are different measures.


### Query 5: Products Above an Average Transaction Threshold Using HAVING

This query demonstrates `GROUP BY HAVING` by returning products whose average transaction value exceeds KES 1,550.


In [8]:
query_5 = '''
SELECT
    product,
    COUNT(txn_id) AS transactions,
    ROUND(AVG(amount), 2) AS average_transaction_value
FROM transactions
GROUP BY product
HAVING AVG(amount) > 1550
ORDER BY average_transaction_value DESC;
'''

having_sql = pd.read_sql_query(
    query_5,
    conn
)

display(having_sql)


,product,transactions,average_transaction_value
0,Merchant Pay,12681,1590.56
1,Bill Pay,12861,1578.13
2,Send Money,21043,1567.21
3,Savings,7047,1563.24
4,Airtime,17192,1554.03


### Query 6: Highest-Value Customers with a Window Ranking

This query calculates customer value and uses the `RANK()` window function. The result is used internally for analysis and is not displayed in the public dashboard.


In [9]:
query_6 = '''
WITH customer_value AS (
    SELECT
        u.user_id,
        u.acquisition_channel,
        u.county,
        COUNT(t.txn_id) AS transactions,
        SUM(t.amount) AS total_value
    FROM users AS u
    JOIN transactions AS t
        ON u.user_id = t.user_id
    GROUP BY
        u.user_id,
        u.acquisition_channel,
        u.county
),
ranked AS (
    SELECT
        *,
        RANK() OVER (
            ORDER BY total_value DESC
        ) AS value_rank
    FROM customer_value
)
SELECT *
FROM ranked
WHERE value_rank <= 20
ORDER BY value_rank;
'''

top_customers_sql = pd.read_sql_query(
    query_6,
    conn
)

display(top_customers_sql)


,user_id,acquisition_channel,county,transactions,total_value,value_rank
0,1544,Referral,Eldoret,21,78548.0,1
1,6614,Referral,Nairobi,22,75954.0,2
2,1370,Referral,Kisumu,27,69320.0,3
3,2040,Search,Nyeri,19,68950.0,4
4,1865,Search,Nairobi,26,68071.0,5
5,741,Referral,Nairobi,15,63395.0,6
6,3184,Agent,Nairobi,12,63354.0,7
7,6611,Referral,Kisumu,19,63232.0,8
8,3312,Agent,Kisumu,21,62565.0,9
9,5909,Social,Nairobi,13,61681.0,10


## 4. Save SQL Queries

The SQL statements used in the notebook are also saved in `sql/queries.sql` so they can be reviewed independently.


In [10]:
sql_text = f'''-- PesaPap Capstone SQL Queries

-- Query 1: Acquisition channel performance using JOIN and GROUP BY
{query_1.strip()}

-- Query 2: Product performance using GROUP BY
{query_2.strip()}

-- Query 3: Monthly trend using CTE, LAG and running SUM window functions
{query_3.strip()}

-- Query 4: County performance using JOIN and GROUP BY
{query_4.strip()}

-- Query 5: Products above average transaction threshold using HAVING
{query_5.strip()}

-- Query 6: Top customer value using RANK window function
{query_6.strip()}
'''

with open(
    SQL_DIR / "queries.sql",
    "w",
    encoding="utf-8"
) as file:
    file.write(sql_text)

print("Saved:", SQL_DIR / "queries.sql")


Saved: /content/drive/MyDrive/plp-capstone-B-robin-kinyua/sql/queries.sql


## 5. Statistical Analysis

### Business Question

Do Referral users generate different customer value from Social users?

### Hypotheses

**Null hypothesis (H₀):** Referral and Social users have the same distribution of customer value.

**Alternative hypothesis (H₁):** Referral and Social users have different distributions of customer value.

Customer value is right-skewed, so the Mann-Whitney U test is used rather than assuming a normal distribution.


In [11]:
customer_analysis = users.merge(
    activity,
    on="user_id",
    how="left"
)

referral_values = customer_analysis.loc[
    customer_analysis["acquisition_channel"] == "Referral",
    "total_value"
]

social_values = customer_analysis.loc[
    customer_analysis["acquisition_channel"] == "Social",
    "total_value"
]

statistic, p_value = mannwhitneyu(
    referral_values,
    social_values,
    alternative="two-sided"
)

print("Referral users:", len(referral_values))
print("Social users:", len(social_values))

print(
    "Referral median customer value:",
    f"KES {referral_values.median():,.2f}"
)

print(
    "Social median customer value:",
    f"KES {social_values.median():,.2f}"
)

print(
    "Mann-Whitney U statistic:",
    f"{statistic:,.2f}"
)

print(
    "p-value:",
    f"{p_value:.6e}"
)

alpha = 0.05

if p_value < alpha:
    print("\nDecision: Reject the null hypothesis.")
else:
    print("\nDecision: Fail to reject the null hypothesis.")


Referral users: 1942
Social users: 1818
Referral median customer value: KES 13,212.50
Social median customer value: KES 11,084.50
Mann-Whitney U statistic: 1,945,284.00
p-value: 6.240670e-08

Decision: Reject the null hypothesis.


### Statistical Test Interpretation

Referral users have a median customer value of **KES 13,212.50**, compared with **KES 11,084.50** for Social users.

The Mann-Whitney U test produces a p-value of approximately **6.24 × 10⁻⁸**, which is far below the 0.05 significance level. Therefore, the null hypothesis is rejected.

Within this synthetic dataset, Referral users have significantly higher customer value than Social users.

This result should be interpreted carefully because the synthetic data generator intentionally includes slightly different engagement probabilities by acquisition channel. The test therefore demonstrates the analytical method and confirms a designed scenario rather than proving a real-world causal effect.


## 6. Business Findings

1. **Referral is the strongest acquisition channel.** It generates approximately KES 29.03 million in transaction value and Referral customers have higher median value than Social customers.

2. **Customer value is concentrated.** The High Value segment contributes approximately 68.95% of total transaction value.

3. **Send Money is the strongest product.** It generates approximately KES 32.98 million from 21,043 transactions.

4. **Nairobi is the largest market by total value**, while smaller counties such as Nyeri show stronger value per customer.

5. **Retention weakens over time.** The July 2025 cohort declines from approximately 66.0% active in month 0 to 32.1% active by month 11.

6. **478 customers are inactive**, while more than 3,000 customers fall into the Low Activity segment.


## 7. Recommendations

1. Increase focus on Referral acquisition while also evaluating real acquisition cost and scalability before reallocating budget.

2. Protect and reward High Value customers because they contribute the majority of transaction value.

3. Develop targeted engagement campaigns for Low Activity customers to increase transaction frequency.

4. Create reactivation campaigns for inactive users.

5. Promote Send Money as a core product while testing cross-sell opportunities into Bill Pay, Merchant Pay and Savings.

6. Explore high-value counties such as Nyeri, Nakuru and Machakos for targeted growth opportunities.

7. Introduce lifecycle retention campaigns several months after signup when cohort activity begins to decline.


## 8. Limitations

This project uses synthetic data, so the results demonstrate an analytical workflow rather than actual PesaPap business performance.

The synthetic generator intentionally applies slightly different engagement probabilities by acquisition channel. Therefore, the observed Referral versus Social difference is partly built into the simulated scenario.

The analysis does not include real acquisition cost by channel, customer profitability, income data, credit risk, competitor activity or wider economic conditions.

The active customer rate measures users who transacted at least once during the full observation period. It should not be interpreted as the percentage active in the latest month.

Monthly transaction growth is influenced by the expanding customer base and differing observation periods, so it should not be interpreted as pure organic growth.

The statistical test identifies an association between acquisition channel and customer value but does not establish causation.

Future analysis should combine acquisition cost, customer value and retention to estimate customer lifetime value and return on acquisition spending.


In [12]:
conn.close()

print("Analysis complete.")
print("Final transactions:", total_transactions)
print(
    "Final transaction value:",
    f"KES {total_transaction_value:,.2f}"
)


Analysis complete.
Final transactions: 70824
Final transaction value: KES 111,178,105.00
